# Module 3: Clone, Modify, Deploy

Clone an existing environment, modify it, test locally, and deploy to HF Spaces.

**Time:** ~25 min · **Difficulty:** Intermediate · **GPU:** Not required

> **Note:** The deployment steps (`openenv push`) require a Hugging Face account and token. The local testing steps work without one.

In [1]:
# Install dependencies
!uv pip install -q openenv-core==0.2.2

## 1. Verify the Hosted Echo Environment

First, let's confirm the hosted Echo environment works.

In [ ]:
from echo_env import EchoEnv, CallToolAction

ECHO_URL = "https://openenv-echo-env.hf.space"

with EchoEnv(base_url=ECHO_URL).sync() as env:
    result = env.reset()
    result = env.step(CallToolAction(tool_name="echo_with_length", arguments= {"message": "Hello, OpenEnv!"}))
    print(f"Sent: ping")
    print(f"Received: {result.observation}")
    print(f"This is the standard Echo — it returns exactly what you send.")

## 2. Clone the Echo Environment

Clone the Space repository to get the full source code.

In [ ]:
!git clone https://huggingface.co/spaces/openenv/echo-env echo-env-modified 2>/dev/null || echo "Already cloned"

import os
os.listdir("echo-env-modified")

## 3. Explore the Structure

Every OpenEnv environment follows the same layout.

In [ ]:
!find echo-env-modified -type f -name "*.py" -o -name "*.yaml" -o -name "*.toml" -o -name "Dockerfile" | sort

In [ ]:
# Look at the environment logic
!cat echo-env-modified/echo_env/server/environment.py

## 4. Modify the Environment

Let's make a "Reverse Echo" — instead of echoing back the message, it reverses it.

We'll modify the `step()` method in `environment.py`.

In [ ]:
# Read the current environment.py
env_file = "echo-env-modified/echo_env/server/environment.py"

with open(env_file, "r") as f:
    content = f.read()

print("Original environment.py:")
print(content)

In [ ]:
# Modify: reverse the echo message in the step method
# This is a simple find-and-replace — adapt based on the actual file content

modified = content.replace(
    'action.message',
    'action.message[::-1]',
    1  # Replace only the first occurrence (in step, not elsewhere)
)

with open(env_file, "w") as f:
    f.write(modified)

print("Modified environment.py (step now reverses the message):")
print(modified)

## 5. Test Locally

Start the modified server and connect to it.

> In Colab, we'll start the server as a background process. Locally, you'd run `uv run server` in a separate terminal.

In [ ]:
import subprocess
import time
import sys

# Start the server in the background
server = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "echo_env.server.app:app",
     "--host", "0.0.0.0", "--port", "8000"],
    cwd="echo-env-modified",
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

# Give it time to start
time.sleep(3)
print(f"Server started (PID: {server.pid})")

In [ ]:
# Test the modified environment
with EchoEnv(base_url="http://localhost:8000").sync() as env:
    result = env.reset()

    test_messages = ["Hello", "OpenEnv", "Reverse this!"]
    for msg in test_messages:
        result = env.step(EchoAction(message=msg))
        print(f"Sent: {msg:20s} → Received: {result.observation}")

In [ ]:
# Clean up the server
server.terminate()
server.wait()
print("Server stopped.")

## 6. Deploy to HF Spaces

Once your environment works locally, deploy it with `openenv push`.

```bash
cd echo-env-modified
openenv push --repo-id YOUR_USERNAME/reverse-echo-env
```

Your environment is now live at:
- **API:** `https://YOUR_USERNAME-reverse-echo-env.hf.space`
- **Web UI:** `https://YOUR_USERNAME-reverse-echo-env.hf.space/web`
- **Docs:** `https://YOUR_USERNAME-reverse-echo-env.hf.space/docs`

In [ ]:
# Uncomment and run to deploy (requires HF token)
# !cd echo-env-modified && openenv push --repo-id YOUR_USERNAME/reverse-echo-env

## 7. Connect to Your Deployed Environment

After deployment, install the client and connect:

In [ ]:
# Uncomment after deploying
# !pip install -q git+https://huggingface.co/spaces/YOUR_USERNAME/reverse-echo-env
#
# with EchoEnv(base_url="https://YOUR_USERNAME-reverse-echo-env.hf.space").sync() as env:
#     result = env.reset()
#     result = env.step(EchoAction(message="Deployed!"))
#     print(f"Response from your Space: {result.observation}")

## 8. Docker Deployment (Alternative)

You can also pull and run the Docker image locally:

```bash
# Pull from HF registry (after deploying)
docker pull registry.hf.space/YOUR_USERNAME-reverse-echo-env:latest
docker run -d -p 8001:8000 registry.hf.space/YOUR_USERNAME-reverse-echo-env:latest

# Or build from source
cd echo-env-modified
docker build -t reverse-echo:latest -f server/Dockerfile .
docker run -d -p 8001:8000 reverse-echo:latest
```

Connect the same way:
```python
with EchoEnv(base_url="http://localhost:8001").sync() as env:
    result = env.reset()
```

## Summary

What you did:
1. Cloned an existing environment from HF Spaces
2. Explored its structure (models, client, server)
3. Modified the environment logic (echo → reverse echo)
4. Tested locally with uvicorn
5. Deployed to HF Spaces with `openenv push`

The workflow is always: **clone → modify → test → deploy**.

**Next:** [Module 4](../module-4/README.md) — Building an environment from scratch.